# Notebook 48 - Two more seeds for the two-epoch arm on CIC-IoMT-2024

**Why this notebook.** Notebook 45 ran full recovery (two epochs) at 90% with three seeds. On the deep network the paired HSR gaps of four rules have means within 0.012 of zero, but three-seed intervals reach 0.024 to 0.055, so the deployment audit of the paper (paired gap interval below 0.02) cannot certify them; on the shallow network every rule's interval sits above zero (+0.007 to +0.019). Both statements rest on three seeds. This notebook adds seeds 401 and 503, so every two-epoch cell has five, the same count as notebook 45's cheap pass.

**What it does.** GPU, about one hour. The committed notebook 45 90% structures of the five rules and the dense teacher, both networks, recovered for two full epochs under seeds 401 and 503 exactly as in notebook 45 (same bridge, teachers, class weights, optimiser, recalibration slice, evaluation subsample and natural-prior sample): 24 recovered models. The analysis pools them with notebook 45's 36 two-epoch rows.

**Readings, fixed in stage 2 before any result.** Per network at five seeds: the headroom condition, the audit's admitted set, rule separation and paired calibration; three expectations stated in advance.

In [ ]:
# Stage 1 - bootstrap (GPU runtime)
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, gc, time, copy, itertools, subprocess, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/SABER_Research/saber-ids")
IOMT_REPO = Path("/content/drive/MyDrive/IOMT_Compression_Research/iomt-compression-research")
os.chdir(REPO)
for p in (str(REPO), str(IOMT_REPO / "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()
from src.saber.bridge_iomt import build_bridge
from src.saber.taxonomy import DEFAULT_COST_PROFILES
from src.saber.metrics import full_model_audit
from src.saber.surgery import prune_cnn1d_channels, profile_forward_flops, count_parameters

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
IN32, IN45 = R / "32_iomt_bridge", R / "45_iomt_headroom"
OUT = R / "48_iomt_full_recovery_seeds"; OUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE = REPO / "data/iomt_bridge"; MODEL_DIR = REPO / "models/iomt"          # both git-ignored, written by NB32
for f in ("N45_runs.csv", "N45_structures.csv", "N45_PREREGISTRATION.json"):
    assert (IN45 / f).exists(), f"missing {IN45 / f}; pull the repo"
assert (IN32 / "bridge_manifest.json").exists(), "missing NB32 bridge manifest; pull the repo"
assert (MODEL_DIR / "shallow_teacher_seed0.pt").exists() and (MODEL_DIR / "deep_teacher_seed0.pt").exists(), "NB32 teachers missing under models/iomt"
print("device:", DEVICE, "| torch", torch.__version__)

In [ ]:
# Stage 2 - pre-registration (frozen: rerunning this cell can never change it after its commit)
PREREG = {
    "notebook": "48_iomt_full_recovery_seeds",
    "question": ("on CIC-IoMT-2024 at 90% realised MAC reduction after two epochs of recovery, with five seeds instead of three: does any "
                 "rule carry more deployed risk than the dense reference (headroom), which structures does the paper's deployment audit "
                 "admit, and are the shallow-network residuals real"),
    "motivation": ("notebook 45, three seeds: no headroom on either network; shallow paired gaps +0.007 to +0.019 with every interval "
                   "above zero; deep means -0.004 to +0.012 with intervals reaching 0.024 to 0.055, so the audit (interval below 0.02) "
                   "admitted only Fisher on the deep network on interval width alone"),
    "design": {
        "structures": "the committed notebook 45 90% structures (N45_{arch}_{method}_b90_removed_groups.csv), five rules, both networks, and the dense teacher",
        "recovery": "two full epochs, batch 1,024, Adam 1e-3, cross-entropy with the notebook 32 class-weight exponent of each network; exactly notebook 45's standard regime",
        "new_seeds": [401, 503],
        "pooled_seeds": [101, 211, 307, 401, 503],
        "pooling": "the 36 standard-regime rows of N45_runs.csv (seeds 101, 211, 307) plus the 24 rows of this notebook; paired by seed",
        "recalibration_and_evaluation": "identical to notebook 45 (fixed 51,200-row slice; class-balanced subsample, at most 4,000 rows per class, seed 0; 200,000-row natural-prior sample, seed 2026)",
    },
    "thresholds": {"tau": 0.02, "tau_E": 0.02, "tau_F": 0.15},
    "readings": {
        "H_headroom": "per network: the best rule (lowest mean HSR) has mean paired gap >= 0.02 with the 95% paired t interval above zero (condition (4) of the paper)",
        "A_audit": ("per network and rule, Algorithm 1 of the paper: admitted if mean paired gap < 0.02 and the upper end of its 95% interval "
                    "< 0.02, mean benign escalation and mean attack miss within 0.02 of the dense reference's, and every family's mean "
                    "recall within 0.15 of the dense reference's"),
        "S_separation": "per network: largest mean paired difference between two rules, and the number of Holm-corrected paired contrasts significant at 0.05",
        "C_calibration": "per network and rule: paired ECE (student minus dense) on the balanced subsample and the natural-prior sample; counts better / worse / straddling",
    },
    "expectations": {
        "Y1": "no headroom on either network at five seeds (as at three)",
        "Y2": "on the deep network the audit admits at least three of the five rules at five seeds",
        "Y3": "on the shallow network every rule's five-seed paired-gap interval stays above zero (the residuals are real)",
    },
    "reporting_rule": "every reading reported as measured; a failed expectation changes the paper's text, it is not dropped",
    "no_test_access": True, "no_new_selection": True,
}
pre_path = OUT / "N48_PREREGISTRATION.json"
if pre_path.exists():
    print("pre-registration already exists; left untouched:", pre_path)
else:
    pre_path.write_text(json.dumps(PREREG, indent=2)); print("pre-registration written:", pre_path)
print(json.dumps(PREREG, indent=2))

ARCHS = ["shallow", "deep"]
METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
NEW_SEEDS = PREREG["design"]["new_seeds"]; SEEDS_ALL = PREREG["design"]["pooled_seeds"]
TAU = PREREG["thresholds"]
SUBSET_FRACTION = 0.10; CAL_SEED = 2026; NAT_SEED = 2026; NAT_ROWS = 200_000; STD_EPOCHS = 2; COLLAPSE_GAP = 0.10
MIN_W = 8
NAME = {"random": "random", "magnitude": "magnitude", "taylor": "Taylor", "fisher": "Fisher", "saber_v2": "leverage", "dense": "dense"}

In [ ]:
# Stage 2b - commit the pre-registration before any result exists (skipped automatically once results exist)
def _prereg_committed():
    rel = "results/saber/48_iomt_full_recovery_seeds/N48_PREREGISTRATION.json"
    h = subprocess.run(["git", "log", "-1", "--format=%h", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    dirty = subprocess.run(["git", "status", "--porcelain", "--", rel], capture_output=True, text=True, cwd=REPO).stdout.strip()
    return bool(h) and not dirty


_results_exist = any(p.is_file() and p.name != "N48_PREREGISTRATION.json" for p in OUT.rglob("*"))
if _results_exist:
    assert _prereg_committed(), ("results exist but the pre-registration is not committed, so it can no longer be committed before any "
                                 "result; stop here and report it")
    print("results already exist in", OUT.name, "and the pre-registration is committed; nothing to do here")
else:
    os.chdir(REPO)
    r = subprocess.run(["python", "tools/commit_cell.py",
                        "NB48 pre-registration, committed before any result: two more seeds (401, 503) for NB45's two-epoch arm at 90% on "
                        "CIC-IoMT-2024. The committed NB45 90% structures of five rules and the dense teacher, both networks, recovered two "
                        "epochs exactly as NB45 (24 models), pooled with NB45's three seeds to five. Readings: headroom condition, the paper's "
                        "audit (gap < 0.02 with CI upper end < 0.02, E and M within 0.02, families within 0.15), rule separation, paired "
                        "calibration. Expectations: Y1 no headroom on either network; Y2 deep audit admits >= 3 of 5 rules; Y3 shallow "
                        "intervals stay above zero for every rule. No test access, no new selection."],
                       capture_output=True, text=True)
    print(r.stdout); print(r.stderr)

In [ ]:
# Stage 3 - bridge, teachers, evaluation samples and helpers (as notebooks 34 and 40)
TRAIN_LOADER, VAL_LOADER, CLASS_NAMES, taxonomy, MANIFEST = build_bridge(DATA_CACHE)
N_CLASSES = len(CLASS_NAMES)
FAM = np.asarray(taxonomy.class_to_family_index); FAMILIES = list(taxonomy.families)
BENIGN_FAM = FAMILIES.index("benign")


class CNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.conv = nn.Sequential(nn.Conv1d(1, 64, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(64),
                                  nn.Conv1d(64, 128, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(128))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(128, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2), *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1); self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


ARCH_CLS = {"shallow": CNN1D, "deep": DeepCNN1D}
TEACHERS, TEACHER_ALPHA = {}, {}
for arch, cls in ARCH_CLS.items():
    payload = torch.load(MODEL_DIR / f"{arch}_teacher_seed0.pt", map_location="cpu", weights_only=False)
    m = cls(N_CLASSES); m.load_state_dict(payload["state_dict"]); TEACHERS[arch] = m.to(DEVICE).eval()
    TEACHER_ALPHA[arch] = float(payload["alpha"])

Xv, Yv = VAL_LOADER.dataset.tensors; VAL_Y_ALL = Yv.numpy()
Xt, Yt = TRAIN_LOADER.dataset.tensors; N_TRAIN = len(Xt)
assert N_TRAIN == MANIFEST["n_train"] and len(Xv) == MANIFEST["n_val"], "bridge tensors do not match the manifest"
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000] for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]; FAM_Y = FAM[EX_Y]
_nat = np.random.default_rng(NAT_SEED).permutation(len(Xv))[:NAT_ROWS]
NAT_X = Xv[_nat].to(DEVICE); NAT_Y = VAL_Y_ALL[_nat]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)
COUNTS = np.bincount(Yt.numpy(), minlength=N_CLASSES)


def class_weights(alpha):
    w = np.zeros(N_CLASSES, dtype=np.float64); nz = COUNTS > 0
    w[nz] = 1.0 / np.power(COUNTS[nz], alpha); w[nz] /= w[nz].mean()
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


CLASS_W = {a: class_weights(TEACHER_ALPHA[a]) for a in ARCHS}
_cal_idx = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(CAL_SEED))[:50 * 1024]
CAL_LOADER = torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, _cal_idx.tolist()), batch_size=1024, shuffle=False)


def forward_logits(model, X):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()


def forward_logits_batch_stats(model, X):
    # same weights, BatchNorm normalising by each batch's own statistics; buffers saved and restored (notebooks 34, 39, 40)
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum, m.num_batches_tracked.clone())
             for n, m in model.named_modules() if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved: m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(X[i:i + 8192]).cpu() for i in range(0, len(X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]; m.running_mean.copy_(rm); m.running_var.copy_(rv); m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval(); return out


def softmax_np(z):
    z = z - z.max(axis=1, keepdims=True); e = np.exp(z); return e / e.sum(axis=1, keepdims=True)


def audit_logits(lg, y):
    a = full_model_audit(lg, y, taxonomy, DEFAULT_COST_PROFILES); pr = softmax_np(lg); pred = lg.argmax(axis=1)
    out = {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]), "family_f1": float(a["family_macro_f1"]),
           "fine_f1": float(a["fine_macro_f1"]), "hsr_balanced": float(a["hsr_balanced_soc"]), "hsr_miss": float(a["hsr_miss_sensitive"]),
           "hsr_fatigue": float(a["hsr_alert_fatigue"]), "ece": float(a["ece15"]),
           "conf_minus_acc": float(pr.max(axis=1).mean() - (pred == y).mean())}
    pf = FAM[pred]; fy = FAM[y]
    out.update({f"recall_{FAMILIES[f]}": float((pf[fy == f] == f).mean()) for f in range(len(FAMILIES)) if (fy == f).any()})
    out.update({f"recall_class_{CLASS_NAMES[c]}": float((pred[y == c] == c).mean()) for c in range(N_CLASSES) if (y == c).any()})
    return out


def audit(model, mode="eval"):
    lg = forward_logits(model, EX_X) if mode == "eval" else forward_logits_batch_stats(model, EX_X)
    return audit_logits(lg, EX_Y)


def audit_natural(model):
    a = audit_logits(forward_logits(model, NAT_X), NAT_Y)
    return {k: a[k] for k in ("b2a", "a2b", "family_f1", "fine_f1", "hsr_balanced", "ece", "conf_minus_acc")}


def recalibrate_bn(model, n_batches=50):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.reset_running_stats(); m.momentum = None
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(CAL_LOADER):
            if i >= n_batches: break
            model(xb.to(DEVICE))
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d): m.momentum = 0.1
    model.eval(); return model


def make_minimal_loader(seed):
    sub = torch.randperm(N_TRAIN, generator=torch.Generator().manual_seed(seed))[: int(N_TRAIN * SUBSET_FRACTION)]
    return torch.utils.data.DataLoader(torch.utils.data.Subset(TRAIN_LOADER.dataset, sub.tolist()), batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def make_full_loader(seed):
    return torch.utils.data.DataLoader(TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def recover(model, loader, epochs, arch):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3); lossf = nn.CrossEntropyLoss(weight=CLASS_W[arch])
    for _ in range(epochs):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); lossf(model(xb.to(DEVICE)), yb.to(DEVICE)).backward(); opt.step()
    return model.eval()


ctx = [{"architecture": a, "model": "teacher, no recovery", **audit(m), **{f"nat_{k}": v for k, v in audit_natural(m).items()}} for a, m in TEACHERS.items()]
pd.DataFrame(ctx).to_csv(OUT / "N48_teachers.csv", index=False)
print(pd.DataFrame(ctx)[["architecture", "model", "hsr_balanced", "b2a", "a2b", "family_f1", "fine_f1", "ece", "conf_minus_acc", "nat_b2a", "nat_ece"]].round(4).to_string(index=False))
print("IoMT ready | train", N_TRAIN, "| balanced eval rows", len(_idx), "| natural-prior rows", len(_nat), "| families", FAMILIES)

In [ ]:
# Stage 4 - the committed notebook 45 90% structures, rebuilt and checked against N45_structures.csv
STRUCT45 = pd.read_csv(IN45 / "N45_structures.csv")


def prune_map_of(pairs):
    pm = {}
    for layer, ch in pairs: pm.setdefault(str(layer), []).append(int(ch))
    return {l: sorted(c) for l, c in pm.items()}


def widths(st):
    return "/".join(str(m.out_channels) for _, m in st.named_modules() if isinstance(m, nn.Conv1d))


def student_for(arch, method):
    if method == "dense":
        return copy.deepcopy(TEACHERS[arch]).to(DEVICE)
    rm = pd.read_csv(IN45 / f"N45_{arch}_{method}_b90_removed_groups.csv")
    st, _ = prune_cnn1d_channels(TEACHERS[arch], prune_map_of(zip(rm.module_path.astype(str), rm.channel_index.astype(int))), EXAMPLE_INPUT, minimum_remaining_per_layer=MIN_W)
    return st.to(DEVICE)


for arch in ARCHS:
    for m in METHODS:
        st = student_for(arch, m); ref = STRUCT45[(STRUCT45.architecture == arch) & (STRUCT45.method == m) & np.isclose(STRUCT45.budget, 0.9)].iloc[0]
        assert count_parameters(st) == int(ref.parameters) and widths(st) == str(ref.widths), f"{arch}/{m}: rebuilt structure differs from N45_structures.csv"
        print(f"{arch:7s} {NAME[m]:9s} widths {widths(st):16s} parameters {count_parameters(st):6d}  (matches notebook 45)")
        del st
gc.collect()

In [ ]:
# Stage 5 - two-epoch recovery, seeds 401 and 503, dense reference + five rules, both networks (resumable)
RUNS = OUT / "N48_runs.csv"
rows = pd.read_csv(RUNS).to_dict("records") if RUNS.exists() else []
done = {(r["architecture"], r["method"], int(r["seed"])) for r in rows}


def make_full_loader(seed):
    return torch.utils.data.DataLoader(TRAIN_LOADER.dataset, batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(seed))


def run_one(arch, method, seed):
    # notebook 45's run_one for the standard regime, unchanged
    torch.manual_seed(seed); np.random.seed(seed); t0 = time.time()
    model = student_for(arch, method)
    recover(model, make_full_loader(seed), STD_EPOCHS, arch)
    tr = audit(model); bs = audit(model, "batch"); recalibrate_bn(model); rc = audit(model); nat = audit_natural(model)
    row = {"regime": "standard", "architecture": arch, "budget": 0.0 if method == "dense" else 0.90, "method": method, "seed": seed, "parameters": int(count_parameters(model)),
           "batch_b2a": bs["b2a"], "collapse_as_trained": bool(tr["b2a"] - bs["b2a"] > COLLAPSE_GAP), "seconds": round(time.time() - t0, 1)}
    row.update({f"trained_{k}": v for k, v in tr.items()}); row.update({f"recal_{k}": v for k, v in rc.items()}); row.update({f"nat_{k}": v for k, v in nat.items()})
    del model; gc.collect()
    return row


JOBS = [(a, m, s) for s in NEW_SEEDS for a in ARCHS for m in ["dense"] + METHODS]
for job in JOBS:
    if (job[0], job[1], int(job[2])) in done: continue
    row = run_one(*job); rows.append(row); done.add((job[0], job[1], int(job[2]))); pd.DataFrame(rows).to_csv(RUNS, index=False)
    print(f"{job[0]:7s} {NAME[job[1]]:9s} s{job[2]}: recal HSR {row['recal_hsr_balanced']:.4f} E {row['recal_b2a']:.4f} M {row['recal_a2b']:.4f} "
          f"famF1 {row['recal_family_f1']:.3f} ECE {row['recal_ece']:.3f} | nat ECE {row['nat_ece']:.3f} | {row['seconds']:.0f}s")
print("runs:", len(rows), "of", len(JOBS))

In [ ]:
# Stage 6 - pooled analysis at five seeds and verdict (rules exactly as pre-registered in stage 2)
new = pd.read_csv(RUNS)
assert len(new) == len(JOBS), f"expected {len(JOBS)} runs, found {len(new)}: finish stage 5 first"
old = pd.read_csv(IN45 / "N45_runs.csv"); old = old[old.regime == "standard"]
assert len(old) == 36 and sorted(old.seed.unique()) == [101, 211, 307], "notebook 45 standard rows not as expected"
common_cols = [c for c in new.columns if c in old.columns]
POOL = pd.concat([old[common_cols], new[common_cols]], ignore_index=True); POOL.to_csv(OUT / "N48_pooled_runs.csv", index=False)
assert sorted(POOL.seed.unique()) == SEEDS_ALL
FAMILIES_ = [c.replace("recal_recall_", "") for c in POOL.columns if c.startswith("recal_recall_") and not c.startswith("recal_recall_class_")]


def series(arch, method, col, seeds=SEEDS_ALL):
    q = POOL[(POOL.architecture == arch) & (POOL.method == method) & POOL.seed.isin(seeds)]
    return q.set_index("seed")[col].sort_index()


def paired(arch, method, col="recal_hsr_balanced", seeds=SEEDS_ALL):
    s = series(arch, method, col, seeds); d = series(arch, "dense", col, seeds); common = s.index.intersection(d.index)
    g = (s.loc[common] - d.loc[common]).to_numpy(); n = len(g); hw = stats.t.ppf(0.975, n - 1) * g.std(ddof=1) / np.sqrt(n)
    return {"n": int(n), "gap": float(g.mean()), "gap_lo": float(g.mean() - hw), "gap_hi": float(g.mean() + hw)}


def holm(ps):
    order = np.argsort(ps); out = np.empty(len(ps)); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, ps[i] * (len(ps) - rank))); out[i] = running
    return out


rows_, H = [], {}
for arch in ARCHS:
    d = POOL[(POOL.architecture == arch) & (POOL.method == "dense")]
    for m in METHODS:
        q = POOL[(POOL.architecture == arch) & (POOL.method == m)]
        g5 = paired(arch, m); g3 = paired(arch, m, seeds=[101, 211, 307]); e = paired(arch, m, "recal_ece"); ne = paired(arch, m, "nat_ece")
        e_diff = float(q.recal_b2a.mean() - d.recal_b2a.mean()); m_diff = float(q.recal_a2b.mean() - d.recal_a2b.mean())
        defic = {f: float(d[f"recal_recall_{f}"].mean() - q[f"recal_recall_{f}"].mean()) for f in FAMILIES_}
        admitted = bool(g5["gap"] < TAU["tau"] and g5["gap_hi"] < TAU["tau"] and abs(e_diff) <= TAU["tau_E"] and abs(m_diff) <= TAU["tau_E"] and max(defic.values()) <= TAU["tau_F"])
        rows_.append({"architecture": arch, "method": m, "n": g5["n"], "gap5": g5["gap"], "gap5_lo": g5["gap_lo"], "gap5_hi": g5["gap_hi"],
                      "gap3": g3["gap"], "gap3_lo": g3["gap_lo"], "gap3_hi": g3["gap_hi"], "E": float(q.recal_b2a.mean()), "E_dense": float(d.recal_b2a.mean()), "E_diff": e_diff,
                      "M_diff": m_diff, "max_family_deficit": max(defic.values()), "worst_family": max(defic, key=defic.get), "admitted": admitted,
                      "ece_gap": e["gap"], "ece_gap_lo": e["gap_lo"], "ece_gap_hi": e["gap_hi"], "nat_ece_gap": ne["gap"], "nat_ece_gap_lo": ne["gap_lo"], "nat_ece_gap_hi": ne["gap_hi"],
                      "ece": float(q.recal_ece.mean()), "ece_dense": float(d.recal_ece.mean()), "family_f1": float(q.recal_family_f1.mean()), "family_f1_dense": float(d.recal_family_f1.mean())})
SUM = pd.DataFrame(rows_); SUM.to_csv(OUT / "N48_summary.csv", index=False)

for arch in ARCHS:
    s = SUM[SUM.architecture == arch]; best = s.loc[s.gap5.idxmin()]
    piv = pd.DataFrame({m: series(arch, m, "recal_hsr_balanced") for m in METHODS}).dropna(); con = []
    for a, b in itertools.combinations(METHODS, 2):
        dd = (piv[a] - piv[b]).to_numpy(); con.append({"contrast": f"{a}-{b}", "mean": float(dd.mean()), "p": float(stats.ttest_rel(piv[a], piv[b]).pvalue)})
    ph = holm(np.array([c["p"] for c in con]))
    for c, p in zip(con, ph): c["p_holm"] = float(p); c["significant_holm"] = bool(p < 0.05)
    H[arch] = {"H_headroom": bool(best.gap5 >= TAU["tau"] and best.gap5_lo > 0), "best_rule": best.method,
               "admitted": [m for m in METHODS if bool(s[s.method == m].admitted.iloc[0])],
               "all_intervals_above_zero": bool((s.gap5_lo > 0).all()),
               "S_max_abs_difference": float(max(abs(c["mean"]) for c in con)), "S_n_holm_significant": int(sum(c["significant_holm"] for c in con)), "contrasts": con,
               "C_better": int((s.ece_gap_hi < 0).sum()), "C_worse": int((s.ece_gap_lo > 0).sum()),
               "C_nat_better": int((s.nat_ece_gap_hi < 0).sum()), "C_nat_worse": int((s.nat_ece_gap_lo > 0).sum()),
               "rules": {r.method: {k: (round(float(getattr(r, k)), 4) if isinstance(getattr(r, k), (float, np.floating)) else getattr(r, k))
                                    for k in ("gap5", "gap5_lo", "gap5_hi", "gap3", "gap3_lo", "gap3_hi", "E", "E_dense", "E_diff", "M_diff", "max_family_deficit", "worst_family", "admitted", "ece_gap", "ece_gap_lo", "ece_gap_hi")}
                         for r in s.itertuples()}}
Y = {"Y1_no_headroom_both": bool(not H["shallow"]["H_headroom"] and not H["deep"]["H_headroom"]),
     "Y2_deep_audit_admits_at_least_three": bool(len(H["deep"]["admitted"]) >= 3),
     "Y3_shallow_all_intervals_above_zero": bool(H["shallow"]["all_intervals_above_zero"])}
verdict = {"notebook": "48_iomt_full_recovery_seeds", "by_architecture": H, "expectations": Y, "new_runs": int(len(new)),
           "collapses_as_trained_new": int(new.collapse_as_trained.sum()), "prereg": PREREG}
(OUT / "N48_verdict.json").write_text(json.dumps(verdict, indent=2, default=float))
for arch in ARCHS:
    h = H[arch]
    print(f"\n{arch}: headroom {h['H_headroom']} | admitted by the audit {[NAME[m] for m in h['admitted']]} | all gap intervals above zero {h['all_intervals_above_zero']} | "
          f"max rule difference {h['S_max_abs_difference']:.4f}, Holm-significant contrasts {h['S_n_holm_significant']} | ECE better {h['C_better']} worse {h['C_worse']} (natural {h['C_nat_better']}/{h['C_nat_worse']})")
    for m, r in h["rules"].items():
        print(f"  {NAME[m]:9s} 5 seeds {r['gap5']:+.4f} [{r['gap5_lo']:+.4f}, {r['gap5_hi']:+.4f}] | 3 seeds {r['gap3']:+.4f} [{r['gap3_lo']:+.4f}, {r['gap3_hi']:+.4f}] | "
              f"E {r['E']:.4f} (dense {r['E_dense']:.4f}) M diff {r['M_diff']:+.4f} | worst family {r['worst_family']} {r['max_family_deficit']:+.3f} | admitted {r['admitted']}")
print("\nexpectations:", json.dumps(Y, indent=1))

In [ ]:
# Stage 7 - figure (600 dpi, paper names): paired gap at three and five seeds per rule and network
SUM = pd.read_csv(OUT / "N48_summary.csv")
plt.rcParams.update({"font.family": "serif", "font.serif": ["Liberation Serif", "DejaVu Serif"], "font.size": 7, "axes.labelsize": 7,
                     "xtick.labelsize": 6.5, "ytick.labelsize": 6.5, "legend.fontsize": 6, "pdf.fonttype": 42, "axes.linewidth": 0.6})
fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.7), sharey=True, gridspec_kw={"wspace": 0.08})
for ax, arch in zip(axes, ARCHS):
    s = SUM[SUM.architecture == arch].set_index("method").loc[METHODS]; y = np.arange(len(METHODS))
    ax.axvspan(-0.06, TAU["tau"], color="#EEEEEE", zorder=0); ax.axvline(0, color="#B8B8B8", lw=0.6)
    ax.errorbar(s.gap3, y + 0.15, xerr=[s.gap3 - s.gap3_lo, s.gap3_hi - s.gap3], fmt="o", ms=2.6, color="#999999", mfc="white", elinewidth=0.7, capsize=1.2, label="3 seeds")
    ax.errorbar(s.gap5, y - 0.15, xerr=[s.gap5 - s.gap5_lo, s.gap5_hi - s.gap5], fmt="o", ms=2.6, color="#1F1F1F", elinewidth=0.7, capsize=1.2, label="5 seeds")
    ax.set_yticks(y); ax.set_yticklabels([NAME[m] for m in METHODS]); ax.set_xlim(-0.05, 0.065); ax.set_title(f"{arch} network", fontsize=7, pad=2)
    ax.set_xlabel("paired HSR gap, two epochs"); ax.spines[["top", "right"]].set_visible(False)
axes[0].invert_yaxis(); axes[0].legend(frameon=False, loc="lower right", handlelength=1.0)
fig.savefig(OUT / "N48_seeds.png", dpi=600, bbox_inches="tight"); fig.savefig(OUT / "N48_seeds.pdf", bbox_inches="tight"); plt.show()
print("written ->", OUT)

In [ ]:
# Stage 8 - commit the results (standard commit cell; the message is filled in from the verdict)
v = json.load(open(OUT / "N48_verdict.json"))
pre = subprocess.run(["git", "log", "-1", "--format=%h", "--", "results/saber/48_iomt_full_recovery_seeds/N48_PREREGISTRATION.json"], capture_output=True, text=True, cwd=REPO).stdout.strip() or "see log"
parts = []
for arch in ARCHS:
    h = v["by_architecture"][arch]
    gaps = "; ".join(f"{NAME[m]} {r['gap5']:+.3f} [{r['gap5_lo']:+.3f}, {r['gap5_hi']:+.3f}]" for m, r in h["rules"].items())
    parts.append(f"{arch}: headroom {h['H_headroom']}, audit admits {[NAME[m] for m in h['admitted']]}, five-seed gaps {gaps}; ECE better {h['C_better']} / worse {h['C_worse']} of 5.")
Y = v["expectations"]
msg = (f"NB48 results (pre-registration {pre}): two-epoch arm at 90% on CIC-IoMT-2024 extended to five seeds (new seeds 401, 503; {v['new_runs']} models). "
       + " ".join(parts) + f" Expectations: Y1 {'HELD' if Y['Y1_no_headroom_both'] else 'NOT HELD'}, Y2 {'HELD' if Y['Y2_deep_audit_admits_at_least_three'] else 'NOT HELD'}, "
       f"Y3 {'HELD' if Y['Y3_shallow_all_intervals_above_zero'] else 'NOT HELD'}. As-trained collapses in the new runs: {v['collapses_as_trained_new']}.")
print(msg)
os.chdir(REPO)
r = subprocess.run(["python", "tools/commit_cell.py", msg], capture_output=True, text=True)
print(r.stdout); print(r.stderr)